# 01 · Why Not a Straight Line?

### Recap & why now
Project 5 ended with a working quadcopter and one uncomfortable detail: every step
command saturated the motors. Tell a drone "be two metres to the left" and it goes flat
out, overshoots, and settles — and while it is saturated, the limits are flying it, not
your controller.

The fix is not a better controller. It is a better **question**. This project is about
asking for motion the vehicle can actually produce.

### Learning objectives
1. Explain why a step command is a physically impossible request.
2. Compare a straight ramp, a smooth curve and a step on the same move.
3. Read **peak velocity and acceleration** off a profile before flying it.
4. Connect acceleration to the **tilt** a quadcopter must adopt.
5. State what a trajectory generator is, as an interface.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection used by the 3-D figures.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print numbers with 4 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=4, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Polynomial toolkit, built up over Notebooks 02-05 ===================

def poly_val(c, t, der=0):
    """Value of the polynomial c at time t, or of its `der`-th derivative."""
    out = 0.0
    for i in range(der, len(c)):                   # Terms below `der` differentiate away to zero.
        factor = 1.0
        for k in range(der):
            factor *= (i - k)                      # i(i-1)...(i-der+1), the falling factorial.
        out += c[i]*factor*t**(i - der)
    return out

def deriv_row(n, t, der):
    """Row r with r @ c = the der-th derivative at time t. One CONSTRAINT is one row."""
    r = np.zeros(n)
    for i in range(der, n):
        factor = 1.0
        for k in range(der):
            factor *= (i - k)
        r[i] = factor*t**(i - der)
    return r

def cost_matrix(n, T, der=4):
    """Q with c^T Q c = integral from 0 to T of (der-th derivative)^2 dt."""
    Q = np.zeros((n, n))
    for i in range(der, n):
        for j in range(der, n):
            ci = np.prod([i - k for k in range(der)])
            cj = np.prod([j - k for k in range(der)])
            power = i + j - 2*der + 1               # From integrating t^(i-der) * t^(j-der).
            Q[i, j] = ci*cj*T**power/power
    return Q

NCOEF = 8                                          # Order 7: eight coefficients, eight boundary conditions.
g = 9.81                                           # Gravity, needed whenever we turn acceleration into tilt.
print("polynomial toolkit ready — order %d, %d coefficients per segment per axis" % (NCOEF-1, NCOEF))

## 1 · What a step actually asks for

A step reference changes value instantly. In the instant it changes, the reference
velocity is **infinite** and so is the acceleration.

The drone cannot comply, so it does the only thing available: maximum tilt, maximum
thrust, and an approach governed by its limits. Everything after that — the overshoot,
the saturation, the untidy settle — follows from a request that was never answerable.

In [ ]:
def step_profile(t, T=3.0, distance=2.0):
    """A step: nothing, then everything."""
    return distance*(t >= 0.0), 0.0, 0.0           # Position jumps; the derivatives are undefined.

def ramp_profile(t, T=3.0, distance=2.0):
    """A straight ramp: constant velocity, with a kick at each end."""
    tau = np.clip(t/T, 0, 1)
    return distance*tau, distance/T*(0 < t < T), 0.0

def smooth_profile(t, T=3.0, distance=2.0):
    """A minimum-jerk curve: everything starts and ends at rest."""
    tau = np.clip(t/T, 0, 1)
    s = 10*tau**3 - 15*tau**4 + 6*tau**5
    sd = (30*tau**2 - 60*tau**3 + 30*tau**4)/T
    sdd = (60*tau - 180*tau**2 + 120*tau**3)/T**2
    return distance*s, distance*sd, distance*sdd

t = np.linspace(-0.3, 3.6, 500)
fig, axes = plt.subplots(1, 3, figsize=(13.5, 2.9))
for prof, name, col in [(step_profile, "step", "C3"), (ramp_profile, "ramp", "C1"),
                        (smooth_profile, "smooth", "C0")]:
    vals = np.array([prof(t_) for t_ in t])
    for j in range(3):
        axes[j].plot(t, vals[:, j], color=col, lw=2, label=name)
for j, name in enumerate(["position [m]", "velocity [m/s]", r"acceleration [m/s$^2$]"]):
    axes[j].set_xlabel("time [s]"); axes[j].set_ylabel(name)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2 · Reading a profile before you fly it

The useful habit: look at the peaks before the drone does. A minimum-jerk move of
distance $d$ over time $T$ has

$$v_{\max} = 1.875\,\frac{d}{T}, \qquad a_{\max} = 5.774\,\frac{d}{T^2}$$

Those coefficients come straight from the polynomial, and they are the whole of
trajectory feasibility checking: pick $T$ so that $a_{\max}$ stays inside what the
vehicle can deliver.

In [ ]:
def peaks(distance, T, n=2001):
    """Peak speed and peak acceleration of a minimum-jerk move."""
    tau = np.linspace(0, 1, n)
    v = distance*(30*tau**2 - 60*tau**3 + 30*tau**4)/T          # First derivative.
    a = distance*(60*tau - 180*tau**2 + 120*tau**3)/T**2        # Second derivative.
    return np.abs(v).max(), np.abs(a).max()

print("  distance   time   peak speed   peak accel   implied tilt")
for d_, T_ in [(2.0, 4.0), (2.0, 2.0), (2.0, 1.0), (5.0, 2.0)]:
    v_pk, a_pk = peaks(d_, T_)
    print("  %8.1f m %6.1f s %10.2f m/s %11.2f m/s^2 %12.1f°" %
          (d_, T_, v_pk, a_pk, np.degrees(np.arctan(a_pk/g))))

print("\nThe coefficients are fixed: peak speed is always %.3f d/T and peak acceleration" %
      (peaks(1.0, 1.0)[0]))
print("%.3f d/T^2. So halving the time doubles the speed and QUADRUPLES the acceleration," % peaks(1.0, 1.0)[1])
print("which is the single most useful fact in this project.")

## 3 · Acceleration is tilt

Project 5 established it: a quadcopter cannot push sideways, so horizontal acceleration
is bought with lean, $a = g\tan\theta$.

That makes the acceleration profile a **tilt profile**, and a trajectory's peak
acceleration a claim about how far the drone will lean. A planner that ignores this
produces beautiful curves the vehicle refuses to fly.

In [ ]:
TILT_LIMIT = np.deg2rad(35)                        # The command limit from Project 5's cascade.
a_limit = g*np.tan(TILT_LIMIT)                     # The acceleration it corresponds to.
print("a 35° tilt limit allows %.2f m/s^2 of horizontal acceleration.\n" % a_limit)

print("  a 5 m move in...   peak accel   tilt needed   flyable?")
for T_ in (5.0, 3.0, 2.0, 1.5, 1.0):
    _, a_pk = peaks(5.0, T_)
    tilt = np.degrees(np.arctan(a_pk/g))
    print("  %10.1f s %13.2f %12.1f°   %s" % (T_, a_pk, tilt, "yes" if a_pk <= a_limit else "NO"))

fig, ax = plt.subplots(figsize=(7.4, 3.0))
Ts = np.linspace(0.8, 6.0, 300)
tilts = [np.degrees(np.arctan(peaks(5.0, T_)[1]/g)) for T_ in Ts]
ax.plot(Ts, tilts, color="C0", lw=2.2)
ax.axhline(35, color="C3", ls="--", lw=1.5); ax.text(4.5, 37, "tilt limit", fontsize=9, color="C3")
ax.set_xlabel("time allowed for a 5 m move [s]"); ax.set_ylabel("peak tilt demanded [deg]")
ax.set_ylim(0, 80); ax.set_title("How fast can this move be?")
plt.show()
print("The crossing is at about %.1f s — faster than that and the planner is asking for a tilt" %
      Ts[np.argmax(np.array(tilts) < 35)])
print("the controller is not allowed to give. Notebook 08 automates finding that boundary.")

## 4 · What a trajectory generator is

An interface, and a small one:

```text
                      ref(t)  ->  (p_des, v_des, a_des)
```

Three vectors at every instant: where to be, how fast, and how hard to accelerate. The
controller from Project 5 already accepts exactly this — its position and velocity loops
have feedforward slots that we filled with zeros.

Everything in the next nine notebooks is about computing better values for those three
vectors. Nothing downstream changes.

In [ ]:
def make_reference(waypoints, durations):
    """Chain minimum-jerk segments into the ref(t) interface the controller expects."""
    wp = [np.asarray(w, float) for w in waypoints]
    edges = np.concatenate([[0.0], np.cumsum(durations)])
    def ref(t):
        if t <= 0:         return wp[0].copy(), np.zeros(3), np.zeros(3)
        if t >= edges[-1]: return wp[-1].copy(), np.zeros(3), np.zeros(3)
        i = int(np.searchsorted(edges, t, side="right") - 1)
        T = durations[i]; tau = (t - edges[i])/T
        s = 10*tau**3 - 15*tau**4 + 6*tau**5
        sd = 30*tau**2 - 60*tau**3 + 30*tau**4
        sdd = 60*tau - 180*tau**2 + 120*tau**3
        d = wp[i+1] - wp[i]
        return wp[i] + d*s, d*sd/T, d*sdd/T**2     # Chain rule: each derivative brings a 1/T.
    return ref, float(edges[-1])

waypoints = [(0, 0, 0), (0, 0, 1.5), (2.0, 0, 1.5), (2.0, 2.0, 1.5)]
ref, T_total = make_reference(waypoints, [2.5, 3.0, 3.0])
grid = np.linspace(0, T_total, 400)
P = np.array([ref(t_)[0] for t_ in grid]); A = np.array([ref(t_)[2] for t_ in grid])
print("duration %.1f s, peak speed %.2f m/s, peak acceleration %.2f m/s^2, peak tilt %.1f°" %
      (T_total, max(np.linalg.norm(ref(t_)[1]) for t_ in grid),
       np.linalg.norm(A, axis=1).max(), np.degrees(np.arctan(np.linalg.norm(A[:, :2], axis=1).max()/g))))

fig = plt.figure(figsize=(6.0, 4.2))
ax = fig.add_subplot(111, projection="3d")
ax.plot(P[:, 0], P[:, 1], P[:, 2], color="C0", lw=2.4)
W = np.array(waypoints, float)
ax.plot(W[:, 0], W[:, 1], W[:, 2], "*", color="C3", ms=12)
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("z [m]")
ax.set_title("Three smooth segments"); ax.view_init(elev=24, azim=-62)
plt.show()

## 🧪 Try it yourself

**E1.** A step command and a smooth command both reach the same place. What does the
smooth one buy, if the destination is identical?

**E2.** Find the shortest time in which this drone can move 10 m horizontally without
exceeding a 35° tilt, and check what speed that implies.

In [ ]:
# --- Solution E1 ---
print("E1: headroom, and control. Both arrive, so on 'did it get there' they tie. The difference")
print("    is what is left over on the way. A step demands maximum tilt and saturates the motors,")
print("    so during that stretch the vehicle's LIMITS are flying it — a gust arriving then has")
print("    nothing to push against. A smooth reference uses a fraction of the authority and keeps")
print("    the rest in reserve, which is exactly what feedback needs to reject disturbances.")

# --- Solution E2 ---
def fastest_time(distance, a_max, lo=0.2, hi=20.0, iters=40):
    """Smallest T whose peak acceleration stays within a_max. Bisection, because peaks fall with T."""
    for _ in range(iters):
        mid = 0.5*(lo + hi)
        if peaks(distance, mid)[1] <= a_max: hi = mid
        else:                                lo = mid
    return hi

T_fast = fastest_time(10.0, a_limit)
v_pk, a_pk = peaks(10.0, T_fast)
print("\nE2: 10 m within a 35° tilt takes at least %.2f s." % T_fast)
print("    That implies a peak speed of %.2f m/s and a peak acceleration of %.2f m/s^2 ✔" % (v_pk, a_pk))
print("    Check it against the closed form: T = sqrt(5.774 d / a_max) = %.2f s ✔" %
      np.sqrt(peaks(1.0, 1.0)[1]*10.0/a_limit))
print("    Note that %.2f m/s is fast — most indoor flights are speed-limited long before they" % v_pk)
print("    are tilt-limited, which is why Notebook 07 adds a velocity constraint as well.")

## 🚁 Mini-project: three ways to move two metres

Animate a drone following the step, the ramp and the smooth profile side by side, with
the tilt each one demands shown underneath. The step is not merely untidy — it asks for
a lean no vehicle can produce.

In [ ]:
T_move, dist = 3.0, 2.0
times = np.linspace(0, 4.0, 160)
profiles = {"step": step_profile, "ramp": ramp_profile, "smooth": smooth_profile}
traces = {k: np.array([f(t_, T_move, dist) for t_ in times]) for k, f in profiles.items()}

fig, (ax, ax2) = plt.subplots(2, 1, figsize=(7.2, 4.8), gridspec_kw={"height_ratios": [2, 1]})

def frame(k):
    ax.clear(); ax2.clear()
    for (name, tr), col, y in zip(traces.items(), ["C3", "C1", "C0"], [2, 1, 0]):
        ax.plot([0, dist], [y, y], color="0.85", lw=2)                    # The route.
        ax.plot(tr[k, 0], y, "o", color=col, ms=12)                       # The drone.
        ax.text(-0.45, y, name, fontsize=9, va="center", color=col)
        ax2.plot(times[:k+1], np.degrees(np.arctan(tr[:k+1, 2]/g)), color=col, lw=1.6)
    ax.set_xlim(-0.6, 2.4); ax.set_ylim(-0.6, 2.6); ax.set_yticks([]); ax.set_xlabel("position [m]")
    ax.set_title("t = %4.2f s" % times[k], fontsize=10)
    ax2.axhline(35, color="C3", ls="--", lw=1.2); ax2.axhline(-35, color="C3", ls="--", lw=1.2)
    ax2.set_xlim(0, times[-1]); ax2.set_ylim(-70, 70)
    ax2.set_xlabel("time [s]"); ax2.set_ylabel("tilt demanded [deg]")
    return []

anim = animation.FuncAnimation(fig, frame, frames=len(times), interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** Every autonomous vehicle has this layer, under one name or
> another — trajectory generator, motion profiler, path smoother. Industrial robot
> controllers call it "jerk-limited motion" and it is why a robot arm moves like a dancer
> rather than a hammer. The idea transfers unchanged: a controller can only be as good as
> the reference it is given.

**Where next.** The smooth profile came from a polynomial nobody has justified yet.
Notebook 02 builds polynomials properly and shows where those coefficients come from.